# Classification Programming

### Dataset
Using the Breast Cancer Wisconsin dataset. Goal is to compare logistic regression, Gaussian Naive Bayes, and KNN for predicting whether a case is malignant.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

cancer = load_breast_cancer(as_frame=True)
df = cancer.frame


## Part (a): Inspect the dataset
Reporting the number of observations, number of predictor features, and class counts.

In [2]:
display(df.head())

n_observations = df.shape[0]
n_features = df.shape[1] - 1  # exclude the target column
class_counts = df['target'].value_counts().rename(index={0: 'malignant', 1: 'benign'})

print(f"Number of observations: {n_observations}")
print(f"Number of predictor features: {n_features}")
print("Class counts (original encoding, 0=malignant, 1=benign):")
print(class_counts)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


Number of observations: 569
Number of predictor features: 30
Class counts (original encoding, 0=malignant, 1=benign):
target
benign       357
malignant    212
Name: count, dtype: int64


**Written response:** Number of observations = 569. Number of predictor features = 30. Class counts = 357 benign, 212 malignant

## Part (b): Recode and split the data

In [3]:
df['target'] = 1 - df['target']

X = df.drop(columns=['target'])
y = df['target']

RANDOM_STATE = 42


X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)


X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=RANDOM_STATE, stratify=y_temp
)

print(f"Training set:   {X_train.shape[0]} observations, {y_train.mean():.4f} proportion malignant")
print(f"Validation set: {X_val.shape[0]} observations, {y_val.mean():.4f} proportion malignant")
print(f"Test set:       {X_test.shape[0]} observations, {y_test.mean():.4f} proportion malignant")
print(f"Full dataset:   {len(y)} observations, {y.mean():.4f} proportion malignant")

Training set:   398 observations, 0.3719 proportion malignant
Validation set: 85 observations, 0.3765 proportion malignant
Test set:       86 observations, 0.3721 proportion malignant
Full dataset:   569 observations, 0.3726 proportion malignant


**Written response:** I used `random_state = 42` with a 70% / 15% / 15% train / validation / test split. The proportion of malignant cases was 0.3719 in training, 0.3765 in validation, 0.3721 in test, versus 0.3726. Stratification works by performing the random split separately within each class so that each resulting subset preserves roughly the same class ratio as the original data.

## Part (c): Standardize and fit candidate models

In [4]:
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

models = {}

# Logistic regression
models['Logistic Regression'] = LogisticRegression(max_iter=10000, random_state=RANDOM_STATE).fit(X_train_s, y_train)

# Gaussian Naive Bayes 
models['Gaussian Naive Bayes'] = GaussianNB().fit(X_train, y_train)

# KNN with k = 1, 5, 15 
for k in [1, 5, 15]:
    models[f'KNN (k={k})'] = KNeighborsClassifier(n_neighbors=k).fit(X_train_s, y_train)

print("Fitted candidate models:")
for name in models:
    print(" -", name)

Fitted candidate models:
 - Logistic Regression
 - Gaussian Naive Bayes
 - KNN (k=1)
 - KNN (k=5)
 - KNN (k=15)


**Written response:** The five candidate models were: (1) Logistic Regression, (2) Gaussian Naive Bayes, (3) KNN with k = 1, (4) KNN with k = 5, and (5) KNN with k = 15.

## Part (d): Validation-set comparison and selection

In [5]:
def evaluate(model, X_eval_scaled_or_raw, y_eval):
    preds = model.predict(X_eval_scaled_or_raw)
    proba = model.predict_proba(X_eval_scaled_or_raw)[:, 1]
    cm = confusion_matrix(y_eval, preds)
    prec = precision_score(y_eval, preds)
    rec = recall_score(y_eval, preds)
    f1 = f1_score(y_eval, preds)
    auc = roc_auc_score(y_eval, proba)
    return cm, prec, rec, f1, auc

val_results = {}
for name, model in models.items():
    X_eval = X_val if name == 'Gaussian Naive Bayes' else X_val_s
    cm, prec, rec, f1, auc = evaluate(model, X_eval, y_val)
    val_results[name] = {'confusion_matrix': cm, 'precision': prec, 'recall': rec, 'f1': f1, 'auc': auc}
    print(f"{name}")
    print(cm)
    print(f"  precision={prec:.4f}  recall={rec:.4f}  f1={f1:.4f}  roc_auc={auc:.4f}")
    print()

best_k_name = max(['KNN (k=1)', 'KNN (k=5)', 'KNN (k=15)'], key=lambda n: val_results[n]['f1'])
final_model_name = max(val_results, key=lambda n: val_results[n]['f1'])
print(f"Best KNN by validation F1: {best_k_name}")
print(f"Best overall model by validation F1: {final_model_name}")

Logistic Regression
[[53  0]
 [ 3 29]]
  precision=1.0000  recall=0.9062  f1=0.9508  roc_auc=0.9965

Gaussian Naive Bayes
[[52  1]
 [ 5 27]]
  precision=0.9643  recall=0.8438  f1=0.9000  roc_auc=0.9900

KNN (k=1)
[[50  3]
 [ 3 29]]
  precision=0.9062  recall=0.9062  f1=0.9062  roc_auc=0.9248

KNN (k=5)
[[53  0]
 [ 3 29]]
  precision=1.0000  recall=0.9062  f1=0.9508  roc_auc=0.9779

KNN (k=15)
[[52  1]
 [ 4 28]]
  precision=0.9655  recall=0.8750  f1=0.9180  roc_auc=0.9944

Best KNN by validation F1: KNN (k=5)
Best overall model by validation F1: Logistic Regression


**Written response:** Validation metrics were: Logistic Regression: precision 1.0000, recall 0.9062, F1 0.9508, ROC-AUC 0.9965; Gaussian Naive Bayes: precision 0.9643, recall 0.8438, F1 0.9000, ROC-AUC 0.9900; KNN (k=1):precision 0.9062, recall 0.9062, F1 0.9062, ROC-AUC 0.9248; KNN (k=5): precision 1.0000, recall 0.9062, F1 0.9508, ROC-AUC 0.9779; KNN (k=15):precision 0.9655, recall 0.8750, F1 0.9180, ROC-AUC 0.9944. 
Best KNN value: **k = 5** (highest F1 and precision among the three k values, and a large improvement in ROC-AUC over k = 1). Final classifier: **Logistic Regression**. Logistic Regression tied KNN (k=5) on precision, recall, and F1, but had the highest ROC-AUC (0.9965). This indicates it ranks malignant vs. benign cases most accurately across all decision thresholds, not just the default one.

## Part (e): Final test evaluation

In [6]:
# Logistic Regression
final_model = models['Logistic Regression']

test_preds = final_model.predict(X_test_s)
test_proba = final_model.predict_proba(X_test_s)[:, 1]

test_cm = confusion_matrix(y_test, test_preds)
test_prec = precision_score(y_test, test_preds)
test_rec = recall_score(y_test, test_preds)
test_f1 = f1_score(y_test, test_preds)
test_auc = roc_auc_score(y_test, test_proba)

print("Final model: Logistic Regression")
print("Test confusion matrix:")
print(test_cm)
print(f"precision={test_prec:.4f}  recall={test_rec:.4f}  f1={test_f1:.4f}  roc_auc={test_auc:.4f}")

Final model: Logistic Regression
Test confusion matrix:
[[53  1]
 [ 1 31]]
precision=0.9688  recall=0.9688  f1=0.9688  roc_auc=0.9988


**Written response:** On the test set, the Logistic Regression model achieved precision = 0.9688, recall = 0.9688, F1 = 0.9688, and ROC-AUC = 0.9988, with confusion matrix [[53, 1], [1, 31]]. These test metrics are essentially in line with the validation metrics (F1 0.9508, ROC-AUC 0.9965), suggesting the model generalizes well. One limitation of this evaluation: the test set has only 86 observations, so metrics like precision and recall are each based on a small number of cases and could change largely with a larger dataset.